In [1]:
import IJulia
import Base64

# The julia kernel has built in support for Revise.jl, so this is the 
# recommended approach for long-running sessions:
# https://github.com/JuliaLang/IJulia.jl/blob/9b10fa9b879574bbf720f5285029e07758e50a5e/src/kernel.jl#L46-L51

# Users should enable revise within .julia/config/startup_ijulia.jl:
# https://timholy.github.io/Revise.jl/stable/config/#Using-Revise-automatically-within-Jupyter/IJulia-1

# clear console history
IJulia.clear_history()

fig_width = 7
fig_height = 5
fig_format = :retina
fig_dpi = 96

# no retina format type, use svg for high quality type/marks
if fig_format == :retina
  fig_format = :svg
elseif fig_format == :pdf
  fig_dpi = 96
  # Enable PDF support for IJulia
  IJulia.register_mime(MIME("application/pdf"))
end

# convert inches to pixels
fig_width = fig_width * fig_dpi
fig_height = fig_height * fig_dpi

# Intialize Plots w/ default fig width/height
try
  import Plots

  # Plots.jl doesn't support PDF output for versions < 1.28.1
  # so use png (if the DPI remains the default of 300 then set to 96)
  if (Plots._current_plots_version < v"1.28.1") & (fig_format == :pdf)
    Plots.gr(size=(fig_width, fig_height), fmt = :png, dpi = fig_dpi)
  else
    Plots.gr(size=(fig_width, fig_height), fmt = fig_format, dpi = fig_dpi)
  end
catch e
  # @warn "Plots init" exception=(e, catch_backtrace())
end

# Initialize CairoMakie with default fig width/height
try
  import CairoMakie

  # CairoMakie's display() in PDF format opens an interactive window
  # instead of saving to the ipynb file, so we don't do that.
  # https://github.com/quarto-dev/quarto-cli/issues/7548
  if fig_format == :pdf
    CairoMakie.activate!(type = "png")
  else
    CairoMakie.activate!(type = string(fig_format))
  end
  CairoMakie.update_theme!(resolution=(fig_width, fig_height))
catch e
    # @warn "CairoMakie init" exception=(e, catch_backtrace())
end
  
# Set run_path if specified
try
  run_path = "QzpcVXNlcnNcTmlrb2xhalxHaXRcZ3JhcGgtZXF1aWxpYnJpdW0tcmFkaWF0aXZlLXRyYW5zZmVyXGV4YW1wbGVz"
  if !isempty(run_path)
    run_path = String(Base64.base64decode(run_path))
    cd(run_path)
  end
catch e
  @warn "Run path init:" exception=(e, catch_backtrace())
end


# emulate old Pkg.installed beahvior, see
# https://discourse.julialang.org/t/how-to-use-pkg-dependencies-instead-of-pkg-installed/36416/9
import Pkg
function isinstalled(pkg::String)
  any(x -> x.name == pkg && x.is_direct_dep, values(Pkg.dependencies()))
end

# ojs_define
if isinstalled("JSON") && isinstalled("DataFrames")
  import JSON, DataFrames
  global function ojs_define(; kwargs...)
    convert(x) = x
    convert(x::DataFrames.AbstractDataFrame) = Tables.rows(x)
    content = Dict("contents" => [Dict("name" => k, "value" => convert(v)) for (k, v) in kwargs])
    tag = "<script type='ojs-define'>$(JSON.json(content))</script>"
    IJulia.display(MIME("text/html"), tag)
  end
elseif isinstalled("JSON")
  import JSON
  global function ojs_define(; kwargs...)
    content = Dict("contents" => [Dict("name" => k, "value" => v) for (k, v) in kwargs])
    tag = "<script type='ojs-define'>$(JSON.json(content))</script>"
    IJulia.display(MIME("text/html"), tag)
  end
else
  global function ojs_define(; kwargs...)
    @warn "JSON package not available. Please install the JSON.jl package to use ojs_define."
  end
end


# don't return kernel dependencies (b/c Revise should take care of dependencies)
nothing


In [2]:
using RayTraceHeatTransfer
using GeometryBasics, StaticArrays, Printf
include(joinpath(pkgdir(RayTraceHeatTransfer), "examples", "anisotropic_slab_reference.jl"))

T1, T2 = 1000.0, 500.0                        # hot and cold plate temperatures [K]
κ, σ_s = 0.2, 0.8                             # absorption and scattering coefficients [1/m]: extinction 1, albedo 0.8
NX = 32                                       # number of cells across the slab
gs = (0.0, 0.5, 0.8)                          # asymmetry factors: isotropic, moderate, strongly forward

ψ(q) = q / (ASLAB_σ * (T1^4 - T2^4))          # net flux normalised by the black-plate exchange σ(T1⁴ − T2⁴)

T_ref = Dict{Float64,Vector{Float64}}()       # reference temperature profile for every g
ψ_ref = Dict{Float64,Float64}()               # reference normalised flux for every g
for g in gs
    T, q = anisotropic_slab_equilibrium(κ, σ_s, g, 1.0, T1, T2; Nx = NX)   # slab of thickness 1 m
    T_ref[g] = T
    ψ_ref[g] = ψ(q)
end

In [3]:
W, NX_H = 2.0, 4
verts = SVector(Point2(0.0, 0.0), Point2(W, 0.0), Point2(W, 1.0), Point2(0.0, 1.0))
face  = PolyVolume2D{Float64}(verts, SVector(true, true, true, true), 1, κ, σ_s)
face.epsilon    = [1.0, 0.01, 1.0, 0.01]
face.T_in_w     = [T1, -1.0, T2, -1.0]        # bottom T1, top T2, sides adiabatic
face.q_in_w     = zeros(4)
face.T_in_g     = -1.0
face.q_in_g     = 0.0
face.phase      = HenyeyGreenstein(0.8)
face.reflection = [DiffuseReflection(), SpecularReflection(1.0), DiffuseReflection(), SpecularReflection(1.0)]

mesh = RayTracingDomain2D([face], [(NX_H, NX)]; verbose = false)
mesh.directional_model = AngularBins(16, 4);  # 16 azimuthal × 4 out-of-plane direction bins

In [4]:
mesh(4 * 10^7; method = :pathlength, chunk_rays = 4 * 10^7, verbose = false)   # paths kept (≈ 4 GB), re-binnable
smooth!(mesh; verbose = false)                                                 # G_smooth, and F_smooth as its sum over bins

# The cavity emulates a 1D slab, so its four columns of cells are four copies of
# the same temperature profile. `slab_result` averages them into one temperature
# per row, and reads the net heat flux off the hot plate.
function slab_result(mesh)
    T_sum   = zeros(NX)                          # summed gas temperature of each row of cells
    n_cells = zeros(Int, NX)                     # number of cells in each row (one per column)
    for cell in mesh.fine_mesh[1]                # all cells of the first (and only) coarse face
        y   = cell.midPoint[2]                   # height of the cell centre, 0 < y < 1
        row = clamp(floor(Int, y * NX) + 1, 1, NX)   # row index, 1 at the hot plate, NX at the cold plate
        T_sum[row]   += cell.T_g                 # T_g: gas temperature written by the solver
        n_cells[row] += 1
    end
    T_profile = T_sum ./ n_cells                 # column-averaged temperature of every row

    q_hot = 0.0                                  # net radiative power leaving the hot plate [W per m depth]
    width = 0.0                                  # length of the hot plate [m]
    # surface_mapping lists every solid wall element as (coarse face, cell, wall of that cell)
    for ((i_face, i_cell, i_wall), _) in mesh.surface_mapping
        cell = mesh.fine_mesh[i_face][i_cell]
        p1 = cell.vertices[i_wall]                                   # wall i_wall runs from this vertex ...
        p2 = cell.vertices[mod1(i_wall + 1, length(cell.vertices))]  # ... to the next one, cyclically
        if p1[2] < 1e-9 && p2[2] < 1e-9          # both ends at y = 0: this element belongs to the hot plate
            q_hot += cell.q_w[i_wall]            # q_w: net radiative power of the wall element [W]
            width += cell.area[i_wall]           # area: its length, per unit depth in 2D
        end
    end
    return T_profile, ψ(q_hot / width)           # temperature profile and normalised net flux
end

# Solve the same domain for another asymmetry factor. The phase function enters only
# the solve, so the exchange factors and their smoothing are reused as they are.
function solve_for(mesh, g)
    for cell in mesh.fine_mesh[1]                # after meshing, the descriptors live on the fine cells
        cell.phase = g == 0 ? IsotropicScattering() : HenyeyGreenstein(g)
    end
    solveEquilibrium!(mesh, mesh.F_smooth; verbose = false)
    return slab_result(mesh)
end

T_pkg = Dict{Float64,Vector{Float64}}()       # package temperature profile for every g
ψ_pkg = Dict{Float64,Float64}()               # package normalised flux for every g
for g in gs
    T_pkg[g], ψ_pkg[g] = solve_for(mesh, g)
end

In [5]:
bins = ((8, 2), (16, 4), (32, 8))             # (azimuthal, out-of-plane) bin counts: 16, 64 and 256 directions
flux_error = Dict(g => Float64[] for g in gs) # signed relative flux error for every g, one entry per resolution
max_dT     = Float64[]                        # largest temperature difference at g = 0.8, one entry per resolution

for (n_azimuth, n_polar) in bins
    mesh.directional_model = AngularBins(n_azimuth, n_polar)   # change the angular resolution ...
    exchangeFactors!(mesh; verbose = false)                    # ... and re-bin the kept ray paths, no new trace
    smooth!(mesh; k_ap = 50_000, verbose = false)              # finer bins need more smoothing iterations
    for g in gs
        T_now, ψ_now = solve_for(mesh, g)
        push!(flux_error[g], (ψ_now - ψ_ref[g]) / ψ_ref[g])    # relative flux error, with its sign
        g == 0.8 && push!(max_dT, maximum(abs.(T_now .- T_ref[g])))
    end
end

In [6]:
using Plots

colours = Dict(0.0 => :black, 0.5 => :blue, 0.8 => :red)      # one colour per asymmetry factor

# Temperature profiles, reference as lines and package as markers
x_cells = ((1:NX) .- 0.5) ./ NX                                # cell-centre positions x / L
p1 = Plots.plot(; xlabel = "x / L", ylabel = "Temperature / K", title = "Slab temperature profile")
for g in gs
    Plots.plot!(p1, x_cells, T_ref[g]; color = colours[g], linewidth = 2, label = "reference, g = $g")
    Plots.scatter!(p1, x_cells, T_pkg[g]; color = colours[g], markersize = 3, label = "16×4 bins, g = $g")
end

In [7]:
# Size of the flux error against the number of direction bins, on logarithmic axes
n_directions = [n_azimuth * n_polar for (n_azimuth, n_polar) in bins]
p2 = Plots.plot(; xscale = :log10, yscale = :log10, xlabel = "direction bins", ylabel = "|Δψ| / ψ",
                  title = "Flux error vs angular resolution", legend = :bottomleft)
for g in gs
    Plots.plot!(p2, n_directions, abs.(flux_error[g]); color = colours[g], marker = :circle, label = "g = $g")
end

In [8]:
#| echo: false
#| output: asis
println("| bins | g = 0 | g = 0.5 | g = 0.8 | max ΔT at g = 0.8 |")
println("|------|------:|--------:|--------:|------------------:|")
for (k, (n_azimuth, n_polar)) in enumerate(bins)
    errors = join((@sprintf("%+.2f %%", 100 * flux_error[g][k]) for g in gs), " | ")
    println("| $(n_azimuth)×$(n_polar) | ", errors, " | ", @sprintf("%.2f K", max_dT[k]), " |")
end

| bins | g = 0 | g = 0.5 | g = 0.8 | max ΔT at g = 0.8 |
|------|------:|--------:|--------:|------------------:|
| 8×2 | 

-0.88 % | -2.67 % | -3.15 % | 1.87 K |
| 16×4 | -0.33 % | -0.96 % | -1.30 % | 1.53 K |
| 32×8 | -0.15 % | -0.36 % | -0.52 % | 0.78 K |
